> 在线讲义：[第 46 章 · 分布式训练账本：显存四项、ZeRO 分片、激活显存、all-reduce 通信量、流水线气泡](https://wikieden.github.io/math-and-eval-stats/lessons/46-distributed-ledger.html) · 本书《数学与实验统计》

# 第 46 章 · 分布式训练账本：显存四项、ZeRO 分片、激活显存、all-reduce 通信量、流水线气泡

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/46-distributed-ledger.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 16 字节 | 16Ψ | `model_states(7.5e9, 1, 0)` | 120 GB |
| 第二步 ZeRO | 三阶段 | `model_states(7.5e9, 64, st)` | 31.4、16.6、1.9 GB |
| 第三步 激活 | sbh(34 + 5as/h) | `activation_bytes(8192, 1, 4096, 32)` | 11.88 GB |
| 第四步 all-reduce | 2(N−1)M/N | `ring_allreduce_bytes(14e9, 64)` | 27.56 GB |
| 第五步 气泡 | (p−1)/(m+p−1) | `bubble(4, 16)` | 15.8% |

例子：混合精度 Adam 的“每个参数 16 字节”；ZeRO 论文图 1 的 7.5B 模型、64 卡；Transformer 一层的激活显存（Korthikanti 2022）；
ring all-reduce 的通信量；流水线气泡。
里程碑 ME-M28 的“显存与通信量计算器（参数、优化器状态、ZeRO 1/2/3、TP / PP），和 Playbook 表对数”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `model_states`**：ZeRO 论文 §3：混合精度 Adam，参数 fp16 2Ψ + 梯度 fp16 2Ψ + 优化器状态 12Ψ（fp32 主权重、m、v）= 16Ψ 字节。

In [ ]:
def model_states(psi: float, nd: int, stage: int) -> float:
    """ZeRO 论文 §3：混合精度 Adam，参数 fp16 2Ψ + 梯度 fp16 2Ψ + 优化器状态 12Ψ（fp32 主权重、m、v）= 16Ψ 字节。
    stage 1 分片优化器状态，2 再分片梯度，3 再分片参数。返回每卡字节数。"""
    p, g, o = 2 * psi, 2 * psi, 12 * psi
    if stage >= 1:
        o /= nd
    if stage >= 2:
        g /= nd
    if stage >= 3:
        p /= nd
    return p + g + o

**函数 `activation_bytes`**：一层 Transformer 的激活显存（不重算、不做序列并行）：s·b·h·(34 + 5·a·s/h) 字节（Korthikanti 2022 式 (2)）；

In [ ]:
def activation_bytes(s: int, b: int, h: int, a: int, tp: int = 1) -> float:
    """一层 Transformer 的激活显存（不重算、不做序列并行）：s·b·h·(34 + 5·a·s/h) 字节（Korthikanti 2022 式 (2)）；
    张量并行 t 时大部分除以 t：s·b·h·(10 + 24/t + 5·a·s/(h·t))。"""
    if tp == 1:
        return s * b * h * (34 + 5 * a * s / h)
    return s * b * h * (10 + 24 / tp + 5 * a * s / (h * tp))

**函数 `ring_allreduce_bytes`**：ring all-reduce：reduce-scatter + all-gather，每张卡发送 2(n − 1)/n·M 字节。

In [ ]:
def ring_allreduce_bytes(m: float, n: int) -> float:
    """ring all-reduce：reduce-scatter + all-gather，每张卡发送 2(n − 1)/n·M 字节。"""
    return 2 * (n - 1) / n * m

**函数 `bubble`**：GPipe / 1F1B 的流水线气泡比例 (p − 1)/(m + p − 1)，p 是流水线级数，m 是微批数。

In [ ]:
def bubble(p: int, m: int) -> float:
    """GPipe / 1F1B 的流水线气泡比例 (p − 1)/(m + p − 1)，p 是流水线级数，m 是微批数。"""
    return (p - 1) / (m + p - 1)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
gb = 1e9

In [ ]:
print("\n第一步 模型状态：每个参数 16 字节")
print(f"  7.5B 参数：{model_states(7.5e9, 1, 0) / gb:.0f} GB（还没算激活）")

In [ ]:
print("\n第二步 ZeRO 分片（64 卡）")
for st, name in ((0, "数据并行"), (1, "ZeRO-1（优化器状态）"), (2, "ZeRO-2（+ 梯度）"), (3, "ZeRO-3（+ 参数）")):
    print(f"  {name}：每卡 {model_states(7.5e9, 64, st) / gb:.1f} GB")

In [ ]:
print("\n第三步 激活：随序列长度平方增长的那一项")
for s in (2048, 8192):
    print(f"  s = {s}、b = 1、h = 4096、32 个头：一层 {activation_bytes(s, 1, 4096, 32) / gb:.2f} GB；"
          f"TP = 8 时 {activation_bytes(s, 1, 4096, 32, 8) / gb:.2f} GB")

In [ ]:
print("\n第四步 通信：ring all-reduce")
for n in (2, 8, 64):
    print(f"  {n} 卡同步 14 GB 梯度（7B、bf16）：每卡发送 {ring_allreduce_bytes(14 * gb, n) / gb:.2f} GB")

In [ ]:
print("\n第五步 流水线气泡")
for p, m in ((4, 4), (4, 16), (8, 64)):
    print(f"  {p} 级、{m} 个微批：气泡 {bubble(p, m):.1%}")

## 3. 代码里才看得到的

1. **GB 用的是 10⁹**：为了和 ZeRO 论文的数字对上；显存条通常按 GiB（2³⁰）标，差 7%。
2. **激活公式不含 embedding 和输出层**：只是一层 Transformer block；完整模型要乘层数再加上首尾两层。
3. **没有做成计算器**：把参数、ZeRO 阶段、TP / PP 组合起来算每卡显存和通信量，并和 Playbook 的表对数，留给自己做（里程碑 ME-M28）。

## 4. 自己改一改

1. 如果优化器换成 Lion（没有二阶矩 v），每个参数是多少字节？ZeRO-1 在 64 卡上每卡多少？
2. 把 `activation_bytes` 里注意力那一项去掉（相当于用了 FlashAttention），s = 8192 时还剩多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 每个参数十六字节：二参二梯十二优化器；
> ZeRO 一二三，依次切优化器、梯度、参数；
> 激活 sbh 乘 (34 + 5as/h)，长序列靠 Flash 和重算；
> ring all-reduce 每卡约 2M；
> 气泡 (p−1)/(m+p−1)，微批多才小。